
**Goal of this notebook:**
1. Reload our SQLite DB and FAISS index from notebook 01
2. Wrap `run_query` and `vector_search` as **LangChain Tools**
3. Build a lightweight **MCP-style Tool Registry** (the Interface Layer)
4. Test that an LLM agent can **autonomously decide which tool to call** and get back structured results

In [ ]:
import os
import sqlite3
import pickle
import json
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from openai import OpenAI

from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_classic.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# ─── CONFIG 
OPENAI_API_KEY=""
DB_PATH        = Path("./mira_data/mimic.db")
FAISS_PATH     = Path("./mira_data/medical_faiss.index")
META_PATH      = Path("./mira_data/faiss_metadata.pkl")
SCHEMA_PATH    = Path("./mira_data/db_schema.txt")
# ───────────

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ Imports done")

✅ Imports done


In [52]:
# Reloading the datalayer from notebook 1 

# ── SQLite connection 
conn = sqlite3.connect(DB_PATH)
DB_SCHEMA = SCHEMA_PATH.read_text()
print("✅ SQLite connected")

# ── FAISS index + metadata 
index = faiss.read_index(str(FAISS_PATH))
with open(META_PATH, "rb") as f:
    MEDICAL_GUIDELINES = pickle.load(f)
print(f"✅ FAISS index loaded: {index.ntotal} vectors")

# ── OpenAI embedding helper 
def get_openai_embeddings(texts: list[str]) -> np.ndarray:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return np.array([r.embedding for r in response.data], dtype=np.float32)

print("✅ Data layer ready")

✅ SQLite connected
✅ FAISS index loaded: 11 vectors
✅ Data layer ready


In [53]:
# We will use langchain's @tool decorator to make plain python functions into callable tools for agents.
# We have created tools for the agents here basically.
@tool
def sql_query(query: str) -> str:
    """
    Execute a SQL SELECT query against the MIMIC-IV patient database.
    Use this tool to retrieve live patient data including:
    - Patient demographics (table: patients — columns: subject_id, gender, anchor_age)
    - Hospital admissions (table: admissions — columns: subject_id, hadm_id, admittime, diagnosis, admission_type)
    - Lab results (table: labevents — columns: subject_id, itemid, value, valueuom, flag, charttime)
    - Lab item names (table: d_labitems — columns: itemid, label)
    - ICD diagnoses (table: diagnoses_icd — columns: subject_id, hadm_id, icd_code, icd_version)
    
    Always JOIN d_labitems ON labevents.itemid = d_labitems.itemid to get human-readable lab names.
    Filter abnormal labs using: WHERE labevents.flag = 'abnormal'
    Returns results as a JSON string. Returns an error message if query fails.
    """
    try:
        df = pd.read_sql_query(query, conn)
        if df.empty:
            return json.dumps({"result": "No data found for this query."})
        # Limit output to avoid overwhelming the LLM context
        result = df.head(20).to_dict(orient="records")
        return json.dumps({"rows": result, "total_returned": len(result)}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e), "hint": "Check table/column names against the schema."})


@tool
def vector_search(query: str, k: int = 3) -> str:
    """
    Search the medical knowledge base (clinical guidelines and literature) using semantic similarity.
    Use this tool to find relevant medical guidelines, treatment protocols, or diagnostic criteria for:
    - Abnormal lab values (high creatinine, low sodium, high glucose, etc.)
    - Clinical conditions (sepsis, AKI, ARDS, heart failure, etc.)
    - Treatment recommendations and safety thresholds
    
    Input should be a clinical description or condition name.
    Returns top-k most relevant guideline chunks with source citations.
    """
    try:
        query_vec = get_openai_embeddings([query])
        distances, indices = index.search(query_vec, k)
        results = []
        for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
            chunk = MEDICAL_GUIDELINES[idx].copy()
            chunk["rank"] = rank + 1
            chunk["relevance_score"] = round(1 / (1 + float(dist)), 4)
            results.append(chunk)
        return json.dumps({"guidelines": results, "query": query}, default=str)
    except Exception as e:
        return json.dumps({"error": str(e)})


# Register both tools
MIRA_TOOLS = [sql_query, vector_search] # Here we are registering the tools for the agents to use.

print("✅ Tools registered:")
for t in MIRA_TOOLS:
    print(f"   🔧 {t.name} — {t.description[:80]}...")

✅ Tools registered:
   🔧 sql_query — Execute a SQL SELECT query against the MIMIC-IV patient database.
Use this tool ...
   🔧 vector_search — Search the medical knowledge base (clinical guidelines and literature) using sem...


In [54]:
# Now in this cell we will make the MCP , it will act as a bridge, from agents to sql database and vector search.
# All tools and everything will be binded in the MCP. Agents can call MCP and MCP will call the tools. MCP will also have the system prompt for the agents to use.
class MIRAToolRegistry:
    """
    MCP-style tool registry — the Interface Layer between agents and data sources.
    Agents don't call SQLite or FAISS directly; they go through this registry.
    """

    def __init__(self, tools: list, db_schema: str):
        self.tools = {t.name: t for t in tools}
        self.db_schema = db_schema
        self.call_log = []  # track every tool call for debugging

    def get_tool_names(self) -> list[str]:
        return list(self.tools.keys())

    def get_tools_list(self) -> list:
        """Returns tool objects for LangGraph agent binding."""
        return list(self.tools.values())

    def get_system_context(self) -> str:
        """Returns the full context string injected into every agent's system prompt."""
        # Escape curly braces in db_schema so ChatPromptTemplate doesn't treat them as variables
        escaped_schema = self.db_schema.replace("{", "{{").replace("}", "}}")
        return f"""
You are a clinical AI assistant with access to two data sources:

1. LIVE PATIENT DATABASE (use sql_query tool)
{escaped_schema}

2. MEDICAL GUIDELINES (use vector_search tool)
   A semantic index of clinical guidelines from:
   - Surviving Sepsis Campaign 2021
   - KDIGO AKI Guidelines
   - ADA Standards of Care 2024
   - ACC/AHA Heart Failure Guidelines
   - ARDSNET Protocol
   - UpToDate Electrolyte Management

Available tools: {self.get_tool_names()}
Always cite your sources. Never guess lab values — query the database.
"""

    def call(self, tool_name: str, **kwargs) -> str:
        """Route a tool call and log it."""
        if tool_name not in self.tools:
            return json.dumps({"error": f"Unknown tool: {tool_name}"})
        result = self.tools[tool_name].invoke(kwargs)
        self.call_log.append({"tool": tool_name, "args": kwargs, "result_preview": result[:100]})
        return result

    def print_call_log(self):
        print(f"\n📋 Tool Call Log ({len(self.call_log)} calls):")
        for i, entry in enumerate(self.call_log):
            print(f"  [{i+1}] {entry['tool']}({entry['args']})")
            print(f"       → {entry['result_preview']}...")


# Instantiate the registry
mcp = MIRAToolRegistry(tools=MIRA_TOOLS, db_schema=DB_SCHEMA)

print("✅ MCP Tool Registry initialized")
print(f"   Registered tools: {mcp.get_tool_names()}")

✅ MCP Tool Registry initialized
   Registered tools: ['sql_query', 'vector_search']


In [55]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system", mcp.get_system_context()),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

agent = create_openai_tools_agent(llm, mcp.get_tools_list(), prompt)
agent_executor = AgentExecutor(
    agent=agent,
    tools=mcp.get_tools_list(),
    verbose=True,          # shows every tool call + result in output
    max_iterations=5,      # safety cap
    return_intermediate_steps=True
)

print("✅ Agent ready with tools:", mcp.get_tool_names())

✅ Agent ready with tools: ['sql_query', 'vector_search']


In [56]:
response = agent_executor.invoke({
    "input": "Which patients have abnormal lab results? Give me their subject_id, age, gender, the lab name, value and flag."
})

print("\n" + "=" * 60)
print("🤖 AGENT FINAL ANSWER:")
print("=" * 60)
print(response["output"])



> Entering new AgentExecutor chain...



Invoking: `sql_query` with `{'query': "SELECT p.subject_id, p.anchor_age, p.gender, d.label AS lab_name, le.value, le.flag FROM patients p JOIN labevents le ON p.subject_id = le.subject_id JOIN d_labitems d ON le.itemid = d.itemid WHERE le.flag = 'abnormal'"}`


{"rows": [{"subject_id": 10014354, "anchor_age": 60, "gender": "M", "lab_name": "Red Blood Cells", "value": "3.35", "flag": "abnormal"}, {"subject_id": 10014354, "anchor_age": 60, "gender": "M", "lab_name": "RDW-SD", "value": "49.7", "flag": "abnormal"}, {"subject_id": 10014354, "anchor_age": 60, "gender": "M", "lab_name": "White Blood Cells", "value": "20.3", "flag": "abnormal"}, {"subject_id": 10014354, "anchor_age": 60, "gender": "M", "lab_name": "MCHC", "value": "31.1", "flag": "abnormal"}, {"subject_id": 10014354, "anchor_age": 60, "gender": "M", "lab_name": "Hematocrit", "value": "29.6", "flag": "abnormal"}, {"subject_id": 10014354, "anchor_age": 60, "gender": "M", "lab_name": "Hemoglobin", "value": "9.2", "flag": "abnor

In [57]:
response = agent_executor.invoke({
    "input": "Find a patient with abnormal creatinine levels. Then search the medical guidelines for how to manage elevated creatinine and what it indicates clinically."
})

print("\n" + "=" * 60)
print("🤖 AGENT FINAL ANSWER:")
print("=" * 60)
print(response["output"])

# Show intermediate steps (which tools were called)
print("\n" + "=" * 60)
print("🔍 INTERMEDIATE STEPS (tool calls made):")
print("=" * 60)
for i, step in enumerate(response["intermediate_steps"]):
    action, observation = step
    print(f"\nStep {i+1}: Called '{action.tool}'")
    print(f"  Input : {str(action.tool_input)[:150]}")
    print(f"  Output: {str(observation)[:200]}...")



> Entering new AgentExecutor chain...

Invoking: `sql_query` with `{'query': "SELECT * FROM labevents JOIN d_labitems ON labevents.itemid = d_labitems.itemid WHERE d_labitems.label = 'Creatinine' AND labevents.flag = 'abnormal' LIMIT 1"}`


{"rows": [{"labevent_id": 172080, "subject_id": 10014354, "hadm_id": 29600294.0, "specimen_id": 43195817, "itemid": 50912, "order_provider_id": null, "charttime": "2148-08-16 00:00:00", "storetime": "2148-08-16 01:47:00", "value": "1.8", "valuenum": 1.8, "valueuom": "mg/dL", "ref_range_lower": 0.5, "ref_range_upper": 1.2, "flag": "abnormal", "priority": "ROUTINE", "comments": null, "label": "Creatinine", "fluid": "Blood", "category": "Chemistry"}], "total_returned": 1}

C:\Users\jenil\AppData\Local\Temp\ipykernel_21336\3382685263.py:23: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  result = df.head(20).to_dict(orient="records")



Invoking: `vector_search` with `{'query': 'elevated creatinine management'}`


{"guidelines": [{"source": "KDIGO AKI Guidelines 2012 (updated 2024)", "topic": "AKI \u2014 Management", "text": "AKI management: Discontinue nephrotoxic agents. Ensure adequate volume status and perfusion pressure. Monitor serum creatinine and urine output. Avoid hyperglycemia. Consider renal replacement therapy (RRT) for volume overload refractory to diuretics, severe hyperkalemia (K\u207a >6.5 mEq/L), metabolic acidosis (pH <7.1), or uremic symptoms. Target MAP \u226565 mmHg in vasopressor-dependent AKI.", "rank": 1, "relevance_score": 0.5382}, {"source": "KDIGO AKI Guidelines 2012 (updated 2024)", "topic": "AKI \u2014 Definition and Staging", "text": "Acute Kidney Injury (AKI) is defined as any of: increase in serum creatinine by \u22650.3 mg/dL within 48 hours, increase in serum creatinine to \u22651.5 times baseline within 7 days, or urine volume <0.5 mL/kg/h for \u22656 hours. KDIGO Stages: Stage 1 \

In [58]:
response = agent_executor.invoke({
    "input": "Get the blood pressure readings for all patients from the vitals table."
    # 'vitals' table doesn't exist in our DB — agent should recover and correct itself
})

print("\n" + "=" * 60)
print("🤖 AGENT RESPONSE (error recovery test):")
print("=" * 60)
print(response["output"])

print("\n🔍 Steps taken:")
for i, step in enumerate(response["intermediate_steps"]):
    action, observation = step
    print(f"  Step {i+1}: {action.tool}({str(action.tool_input)[:100]})")
    print(f"  Result: {str(observation)[:150]}")



> Entering new AgentExecutor chain...

Invoking: `sql_query` with `{'query': "SELECT * FROM vitals WHERE label = 'Blood Pressure'"}`


{"error": "Execution failed on sql 'SELECT * FROM vitals WHERE label = 'Blood Pressure'': no such table: vitals", "hint": "Check table/column names against the schema."}I apologize for the error. It seems there is no table named "vitals" in the database schema. Could you please provide the correct table name where the blood pressure readings are stored?

> Finished chain.

🤖 AGENT RESPONSE (error recovery test):
I apologize for the error. It seems there is no table named "vitals" in the database schema. Could you please provide the correct table name where the blood pressure readings are stored?

🔍 Steps taken:
  Step 1: sql_query({'query': "SELECT * FROM vitals WHERE label = 'Blood Pressure'"})
  Result: {"error": "Execution failed on sql 'SELECT * FROM vitals WHERE label = 'Blood Pressure'': no such table: vitals", "hint": "Check table/column names 

In [59]:
# After all tests above, let's see total tool calls made via the registry
# Note: agent_executor calls tools directly so we log manually here for demo
print("📊 Summary of this notebook session:")
print(f"   Tools available : {mcp.get_tool_names()}")
print(f"   FAISS vectors   : {index.ntotal}")
print(f"   DB tables       : {pd.read_sql_query('SELECT name FROM sqlite_master WHERE type=\'table\'', conn)['name'].tolist()}")
print(f"   Guidelines loaded: {len(MEDICAL_GUIDELINES)}")

print("\n✅ Notebook 02 complete!")
print("\nWhat we built:")
print("  🔧 sql_query tool   → LLM can now query live patient data with natural language")
print("  🔧 vector_search tool → LLM can retrieve clinical guidelines semantically")
print("  🏗️  MIRAToolRegistry  → Central MCP-style interface layer")
print("  🤖 AgentExecutor    → Single agent that autonomously picks and calls tools")
print("\nNext: 03_agent_graph.ipynb — wire 4 specialized agents into a LangGraph pipeline")

📊 Summary of this notebook session:
   Tools available : ['sql_query', 'vector_search']
   FAISS vectors   : 11
   DB tables       : ['patients', 'admissions', 'diagnoses_icd', 'labevents', 'd_labitems']
   Guidelines loaded: 11

✅ Notebook 02 complete!

What we built:
  🔧 sql_query tool   → LLM can now query live patient data with natural language
  🔧 vector_search tool → LLM can retrieve clinical guidelines semantically
  🏗️  MIRAToolRegistry  → Central MCP-style interface layer
  🤖 AgentExecutor    → Single agent that autonomously picks and calls tools

Next: 03_agent_graph.ipynb — wire 4 specialized agents into a LangGraph pipeline
